# Secrets of Strixhaven — Card Ratings Tracker
Load daily 17lands card rating exports and plot how key metrics evolve over time.

## Setup

In [2]:
import os
import re
import glob
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display, clear_output


## Load Data


In [3]:
DATA_DIR = "data" 
RARITY_COLORS = {"C": "#9b9b9b", "U": "#6cb4e4", "R": "#c7a137", "M": "#c97c2e"}
RARITY_ORDER  = ["C", "U", "R", "M"]


def _pct(val):
    """'57.3%' -> 57.3  (or NaN)"""
    if pd.isna(val):
        return np.nan
    s = str(val).strip().rstrip("%")
    try:
        return float(s)
    except ValueError:
        return np.nan


def _pp(val):
    """'4.6pp' -> 4.6  (or NaN)"""
    if pd.isna(val):
        return np.nan
    s = str(val).strip().lower().replace("pp", "")
    try:
        return float(s)
    except ValueError:
        return np.nan


def load_all_files(data_dir):
    pattern = os.path.join(data_dir, "card-ratings-*.csv")
    files   = sorted(glob.glob(pattern))
    if not files:
        raise FileNotFoundError(f"No files matching '{pattern}' found.")

    frames = []
    for path in files:
        m = re.search(r"(\d{4}-\d{2}-\d{2})", os.path.basename(path))
        if not m:
            continue
        date = pd.to_datetime(m.group(1), format="%Y-%m-%d")
        df   = pd.read_csv(path, encoding="utf-8-sig")

        pct_cols = ["% GP", "GP WR", "OH WR", "GD WR", "GIH WR", "GNS WR"]
        for c in pct_cols:
            if c in df.columns:
                df[c] = df[c].apply(_pct)
        if "IIH" in df.columns:
            df["IIH"] = df["IIH"].apply(_pp)

        df.insert(0, "Date", date)
        frames.append(df)

    combined = pd.concat(frames, ignore_index=True)
    combined["Color"]  = combined["Color"].fillna("Colorless")
    combined["Rarity"] = combined["Rarity"].fillna("?")
    return combined


df_all = load_all_files(DATA_DIR)
dates  = sorted(df_all["Date"].unique())
cards  = sorted(df_all["Name"].unique())

print(f"Loaded {len(dates)} day(s) of data across {len(cards)} cards.")
print(f"Date range: {pd.Timestamp(dates[0]).date()} -> {pd.Timestamp(dates[-1]).date()}")
df_all.head(3)

Loaded 3 day(s) of data across 341 cards.
Date range: 2026-04-21 -> 2026-04-23


,Date,Name,Color,Rarity,# Seen,ALSA,# Picked,ATA,# GP,% GP,GP WR,# OH,OH WR,# GD,GD WR,# GIH,GIH WR,# GNS,GNS WR,IIH
0,2026-04-21,The Dawning Archaic,Colorless,M,224,NaN,137,NaN,641,88.3,55.7,95,NaN,199,NaN,294,NaN,330,NaN,NaN
1,2026-04-21,Rancorous Archaic,Colorless,C,9941,7.55,1275,10.73,1725,27.2,53.5,274,NaN,541,59.0,815,56.4,883,50.2,6.3
2,2026-04-21,Sundering Archaic,Colorless,U,3573,5.53,721,7.30,2101,56.0,57.4,361,NaN,657,63.8,1018,61.4,1037,53.2,8.2


## Single-Card Deep Dive
Pick a card and see all of its key metrics plotted over time.

In [4]:
METRICS = {
    "GIH WR (%)": "GIH WR",
    "OH WR (%)":  "OH WR",
    "GD WR (%)":  "GD WR",
    "GNS WR (%)": "GNS WR",
    "GP WR (%)":  "GP WR",
    "IIH (pp)":   "IIH",
    "ALSA":       "ALSA",
    "ATA":        "ATA",
    "% GP":       "% GP",
    "# Seen":     "# Seen",
    "# GIH":      "# GIH",
}

card_dd   = widgets.Combobox(
    options=cards,
    description="Card:",
    placeholder="Start typing...",
    ensure_option=True,
    layout=widgets.Layout(width="380px"),
)
metric_ms = widgets.SelectMultiple(
    options=list(METRICS.keys()),
    value=["GIH WR (%)", "OH WR (%)", "ALSA", "IIH (pp)"],
    description="Metrics:",
    rows=len(METRICS),
    layout=widgets.Layout(width="220px"),
)
btn = widgets.Button(description="Plot", button_style="primary")
out = widgets.Output()


def plot_card(_):
    with out:
        clear_output(wait=True)
        name = card_dd.value
        if not name or name not in cards:
            print("Please select a valid card name.")
            return

        selected = list(metric_ms.value)
        if not selected:
            print("Select at least one metric.")
            return

        sub = df_all[df_all["Name"] == name].sort_values("Date")
        if sub.empty:
            print(f"No data for '{name}'.")
            return

        meta         = sub.iloc[-1]
        rarity_label = meta["Rarity"]
        color_label  = meta["Color"]
        rc           = RARITY_COLORS.get(rarity_label, "#aaaaaa")

        n   = len(selected)
        fig = make_subplots(
            rows=n, cols=1,
            shared_xaxes=True,
            subplot_titles=selected,
            vertical_spacing=0.06,
        )

        for i, label in enumerate(selected, start=1):
            col = METRICS[label]
            if col not in sub.columns:
                continue
            y        = sub[col]
            x        = sub["Date"]
            size_col = "# GIH" if "WR" in col else "# Seen"
            n_games  = sub.get(size_col, pd.Series([None] * len(sub)))

            hover_text = [
                f"<b>{name}</b><br>"
                f"{label}: <b>{v:.2f}</b><br>"
                f"Sample size: {ns:,}"
                for v, ns in zip(y, n_games)
            ]

            fig.add_trace(
                go.Scatter(
                    x=x, y=y,
                    mode="lines+markers",
                    name=label,
                    line=dict(color=rc, width=2.5),
                    marker=dict(size=8, color=rc, line=dict(width=1.5, color="white")),
                    hovertext=hover_text,
                    hoverinfo="text",
                    showlegend=False,
                ),
                row=i, col=1,
            )

            if "WR" in col:
                fig.add_hline(
                    y=50,
                    line=dict(dash="dot", color="rgba(150,150,150,0.5)", width=1),
                    row=i, col=1,
                )
            if col == "IIH":
                fig.add_hline(
                    y=0,
                    line=dict(dash="dot", color="rgba(150,150,150,0.5)", width=1),
                    row=i, col=1,
                )

        fig.update_layout(
            title=dict(
                text=f"<b>{name}</b>  |  {rarity_label}  |  {color_label}",
                font=dict(size=18),
            ),
            height=240 * n + 80,
            hovermode="x unified",
            template="plotly_dark",
            margin=dict(t=80, l=60, r=30, b=40),
        )
        fig.update_xaxes(tickformat="%b %d", tickangle=-30)
        fig.show()


btn.on_click(plot_card)
display(widgets.HBox([widgets.VBox([card_dd, btn]), metric_ms]), out)

Output()

## Snapshot — Latest Day Rankings
Scatter all cards by GIH WR vs. ALSA for the most recent data file. Hover for details.

In [5]:
latest = df_all[df_all["Date"] == df_all["Date"].max()].copy()
latest["Rarity"] = pd.Categorical(latest["Rarity"], categories=RARITY_ORDER, ordered=True)
latest = latest.dropna(subset=["GIH WR", "ALSA"])

fig = px.scatter(
    latest,
    x="ALSA", y="GIH WR",
    color="Rarity",
    color_discrete_map=RARITY_COLORS,
    symbol="Color",
    size="# GIH",
    size_max=24,
    hover_name="Name",
    hover_data={
        "Color": True, "Rarity": True, "# GIH": True,
        "ALSA": ":.2f", "GIH WR": ":.1f", "IIH": ":.1f",
    },
    labels={"GIH WR": "GIH Win Rate (%)", "ALSA": "Avg Last Seen At (ALSA)"},
    title=f"All Cards — GIH WR vs. ALSA  ({pd.Timestamp(latest['Date'].iloc[0]).date()})",
    template="plotly_dark",
)
# Higher ALSA = picked later = lower priority; invert x so best cards sit top-right
fig.update_xaxes(autorange="reversed")
fig.add_hline(y=50, line=dict(dash="dot", color="rgba(200,200,200,0.35)"))
fig.update_layout(height=600, legend=dict(orientation="h", y=-0.15))
fig.show()

## Multi-Card Comparison
Compare up to 8 cards side-by-side over time.

In [6]:
compare_dd = widgets.SelectMultiple(
    options=cards,
    description="Cards:",
    rows=12,
    layout=widgets.Layout(width="360px", height="220px"),
)
compare_metric = widgets.Dropdown(
    options=list(METRICS.keys()),
    value="GIH WR (%)",
    description="Metric:",
    layout=widgets.Layout(width="240px"),
)
compare_btn = widgets.Button(description="Compare", button_style="success")
compare_out = widgets.Output()

PALETTE = px.colors.qualitative.Bold


def plot_compare(_):
    with compare_out:
        clear_output(wait=True)
        names = list(compare_dd.value)[:8]
        if not names:
            print("Select at least one card (Ctrl/Cmd+click for multiple).")
            return

        col   = METRICS[compare_metric.value]
        label = compare_metric.value
        fig   = go.Figure()

        for idx, name in enumerate(names):
            sub = df_all[df_all["Name"] == name].sort_values("Date")
            if sub.empty or col not in sub.columns:
                continue
            color = PALETTE[idx % len(PALETTE)]
            fig.add_trace(go.Scatter(
                x=sub["Date"], y=sub[col],
                mode="lines+markers",
                name=name,
                line=dict(color=color, width=2.5),
                marker=dict(size=8, color=color, line=dict(width=1.5, color="white")),
                hovertemplate=f"<b>{name}</b><br>{label}: %{{y:.2f}}<extra></extra>",
            ))

        if "WR" in col:
            fig.add_hline(y=50, line=dict(dash="dot", color="rgba(200,200,200,0.35)"))
        if col == "IIH":
            fig.add_hline(y=0,  line=dict(dash="dot", color="rgba(200,200,200,0.35)"))

        fig.update_layout(
            title=f"Card Comparison — {label}",
            xaxis_title="Date",
            yaxis_title=label,
            hovermode="x unified",
            template="plotly_dark",
            height=480,
            legend=dict(orientation="h", y=-0.25),
            xaxis=dict(tickformat="%b %d", tickangle=-30),
        )
        fig.show()


compare_btn.on_click(plot_compare)
display(
    widgets.HBox([compare_dd, widgets.VBox([compare_metric, compare_btn])]),
    compare_out,
)

Output()

## Biggest Movers
Which cards changed the most in a given metric between the first and last available day?

In [7]:
mover_metric = widgets.Dropdown(
    options=list(METRICS.keys()),
    value="GIH WR (%)",
    description="Metric:",
    layout=widgets.Layout(width="240px"),
)
top_n_slider = widgets.IntSlider(
    value=15, min=5, max=40, step=5,
    description="Top N:",
    layout=widgets.Layout(width="300px"),
)
mover_btn = widgets.Button(description="Find Movers", button_style="warning")
mover_out = widgets.Output()


def plot_movers(_):
    with mover_out:
        clear_output(wait=True)
        col   = METRICS[mover_metric.value]
        label = mover_metric.value
        n     = top_n_slider.value

        d_first = df_all[df_all["Date"] == df_all["Date"].min()].set_index("Name")[col].rename("first")
        d_last  = df_all[df_all["Date"] == df_all["Date"].max()].set_index("Name")[col].rename("last")
        info    = df_all[df_all["Date"] == df_all["Date"].max()].set_index("Name")[["Rarity", "Color"]]

        delta = pd.concat([d_first, d_last], axis=1).dropna()
        delta["delta"] = delta["last"] - delta["first"]
        delta = delta.join(info)

        top_risers  = delta.nlargest(n, "delta")
        top_fallers = delta.nsmallest(n, "delta")
        movers      = pd.concat([top_risers, top_fallers]).drop_duplicates().sort_values("delta")

        bar_colors = ["#e63946" if v < 0 else "#2a9d8f" for v in movers["delta"]]

        fig = go.Figure(go.Bar(
            y=movers.index,
            x=movers["delta"],
            orientation="h",
            marker_color=bar_colors,
            hovertemplate="<b>%{y}</b><br>Change: %{x:+.2f}<extra></extra>",
            text=[f"{v:+.2f}" for v in movers["delta"]],
            textposition="outside",
        ))
        fig.add_vline(x=0, line=dict(color="white", width=1))

        d0 = pd.Timestamp(df_all["Date"].min()).date()
        d1 = pd.Timestamp(df_all["Date"].max()).date()
        fig.update_layout(
            title=f"Biggest Movers — {label}  ({d0} -> {d1})",
            xaxis_title=f"Change in {label}",
            template="plotly_dark",
            height=max(500, 28 * len(movers) + 120),
            margin=dict(l=220, r=80),
        )
        fig.show()


mover_btn.on_click(plot_movers)
display(widgets.HBox([mover_metric, top_n_slider, mover_btn]), mover_out)

Output()

## Full Data Table (latest day)
Sortable view of all cards from the most recent file.

In [8]:
display_cols = [
    "Name", "Color", "Rarity", "ALSA", "ATA",
    "% GP", "GP WR", "OH WR", "GD WR", "GIH WR", "GNS WR", "IIH", "# GIH",
]

snapshot = (
    df_all[df_all["Date"] == df_all["Date"].max()]
    [[c for c in display_cols if c in df_all.columns]]
    .sort_values("GIH WR", ascending=False)
    .reset_index(drop=True)
)


def fmt(v):
    if isinstance(v, float):
        return f"{v:.2f}" if not pd.isna(v) else ""
    return str(v) if not pd.isna(v) else ""


fig = go.Figure(go.Table(
    header=dict(
        values=[f"<b>{c}</b>" for c in snapshot.columns],
        fill_color="#264653",
        font=dict(color="white", size=12),
        align="left",
    ),
    cells=dict(
        values=[[fmt(v) for v in snapshot[c]] for c in snapshot.columns],
        fill_color=[
            ["#1c2a38" if i % 2 == 0 else "#1e3045" for i in range(len(snapshot))]
            for _ in snapshot.columns
        ],
        font=dict(color="white", size=11),
        align="left",
        height=24,
    ),
))
fig.update_layout(
    title=f"All Cards — {pd.Timestamp(df_all['Date'].max()).date()} (sorted by GIH WR)",
    template="plotly_dark",
    height=min(900, 30 * len(snapshot) + 120),
    margin=dict(t=60, l=10, r=10, b=10),
)
fig.show()